# Memory
This notebook is my exploration of the [Add Memory](https://langchain-ai.github.io/langgraph/tutorials/get-started/3-add-memory/) tutorial by Langgraph. It follows the previous notebooks that create a [simple chatbot without memory](./simple_chatbot_langgraph.ipynb) and [provide a web search tool to the chatbot](./chatbot_with_tools_langgraph.ipynb). To allow multi-turn conversations, just like those in web chatbots like ChatGPT or Claude, we need to save the conversation and pass it as context with each call to the LLM. 

Langgraph supports conversation history through a mechanism called *persistent checkpointing*. To use the mechanism, we need to provide a `checkpointer` and a `thread_id` when initialising the graph. Once provided, when the graph is called for the first time, Langgraph persists the state of the graph on disk after every disk. If we resume the graph with the same `thread_id`, Langgraph automatically picks up the preivous state of the graph and uses it as context in subsequent conversations with the model.

## Required packages
* `langchain[anthropic]`
* `langchain-tavily`
* `langgraph`
* `langgraph-checkpoint-sqlite`
* `langsmith`

## Build a chatbot with tool access
Like we did previously, let us create a chatbot graph that can use the web search tool as needed.

In [14]:
import json
import sqlite3
import uuid
from pprint import pprint
from typing import Annotated
from typing_extensions import TypedDict

from IPython.display import display, Markdown
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.sqlite import SqliteSaver 
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
# Load Anthropic and Tavily API keys
# The `secrets.env` file is expected to have the following lines:
# ANTHROPIC_API_KEY="sk-ant-xxxxx"
# TAVILY_API_KEY="tvly-dev-xxxxx"
from dotenv import load_dotenv
load_dotenv("../../secrets.env")

True

In [3]:
search_tool = TavilySearch(max_results=2)
tools = [search_tool]

In [4]:
llm = init_chat_model("anthropic:claude-sonnet-4-0")

class State(TypedDict):
    messages: Annotated[list, add_messages]

# Tell the LLM the tools it can call
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [5]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

## Memory
We now configure a simple SQLite database to persist the conversation in and then pass the memory object while compiling the graph.

In [ ]:
# Note: check_same_thread=False is OK as the implementation uses a lock
# to ensure thread safety.
conn = sqlite3.connect("chatbot_memory.db", check_same_thread=False)
memory = SqliteSaver(conn)

graph = graph_builder.compile(checkpointer=memory)

## Interacting with the chatbot
Let us now give the chatbot with memory a spin. In order to enable the memory, we need to assign a `thread_id` to our conversation. The configuration dictionary created in this process is passed to the `graph.stream()` method.

In [8]:
config = {"configurable": {"thread_id": uuid.uuid4()}}

In [10]:
def display_text_response(text: str) -> None:
    """Display a Markdown-formatted assistant response."""
    display(Markdown("**Assistant:**\n" + text))

def display_tool_use(tool_info: dict) -> None:
    """Display information about a tool usage in Markdown format.
    
    Args:
        tool_info (dict): Dictionary containing tool usage details.
    """
    tool_name = tool_info.get("name", "")
    tool_input = tool_info.get("input", {})
    query = tool_input.get("query", "")
    topic = tool_input.get("topic", "")
    search_depth = tool_input.get("search_depth", "")
    display(Markdown(
        f"**Tool Use:** `{tool_name}`\n\n"
        f"- **Query:** `{query}`\n"
        f"- **Topic:** `{topic}`\n"
        f"- **Search Depth:** `{search_depth}`\n"
    ))

def display_search_results(results: list) -> None:
    """Display a list of search results in Markdown format.
    
    Args:
        results (list): List of dictionaries containing search result details.
    """
    for result in results:
        url = result.get("url", "")
        title = result.get("title", "")
        score = result.get("score", "")
        published_date = result.get("published_date", "")
        display(Markdown(
            f"**Search Result:**\n"
            f"- **Title:** [{title}]({url})\n"
            f"- **Score:** `{score}`\n"
            f"- **Published Date:** `{published_date}`\n"
        ))

def handle_latest_message(latest_message: object) -> None:
    """Handle and display the latest message from the graph event.
    
    Args:
        latest_message (object): The latest message, can be a list or string.
    """
    if isinstance(latest_message, list):
        for v in latest_message:
            if v["type"] == "text":
                display_text_response(v["text"])
            elif v["type"] == "tool_use":
                display_tool_use(v)
            else:
                raise ValueError("Message type is neither 'text' nor 'tool_use'")
    elif isinstance(latest_message, str):
        try:
            results = json.loads(latest_message)["results"]
            display_search_results(results)
        except json.JSONDecodeError:
            display_text_response(latest_message)
    else:
        raise ValueError("Latest message is neither of 'str' or 'list'")

def stream_graph_updates(user_input: str) -> None:
    """Stream updates from the graph for a given user input and display responses.
    
    Args:
        user_input (str): The user's input message.
    """
    for event in graph.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    ):
        for value in event.values():
            latest_message = value["messages"][-1].content
            handle_latest_message(latest_message)

while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    display(Markdown(f"**Question**:\n{user_input}\n"))
    stream_graph_updates(user_input)

**Question**:
Where was the latest NeurIPS conference held?


**Assistant:**
I'll search for information about the latest NeurIPS conference location.

**Tool Use:** `tavily_search`

- **Query:** `latest NeurIPS conference 2024 location where held`
- **Topic:** ``
- **Search Depth:** `basic`


**Search Result:**
- **Title:** [ELLIS contributions to NeurIPS 2024 - ELLIS Society](https://ellis.eu/contributions-neurips-2024)
- **Score:** `0.9121316`
- **Published Date:** ``


**Search Result:**
- **Title:** [NeurIPS Annual 2024 Conference - Vancouver Convention Centre](https://www.vancouverconventioncentre.com/events/neurips-annual-2024-conference)
- **Score:** `0.9057319`
- **Published Date:** ``


...

**Search Result:**
- **Title:** [2025 Conference](https://neurips.cc/)
- **Score:** `0.98152`
- **Published Date:** ``


**Tool Use:** `tavily_search`

- **Query:** `"NeurIPS 2024" "keynote" OR "invited speakers" program schedule`
- **Topic:** ``
- **Search Depth:** `advanced`


**Search Result:**
- **Title:** [NeurIPS 2024 Daily Schedule](https://nips.cc/virtual/2024/daily-schedule)
- **Score:** `0.7803451`
- **Published Date:** ``


**Search Result:**
- **Title:** [NeurIPS 2024 Invited Talks](https://neurips.cc/virtual/2024/eventlistwithbios/invited%20talk)
- **Score:** `0.76659125`
- **Published Date:** ``


**Assistant:**
Based on my search results, the keynote speakers at NeurIPS 2024 included:

1. **Alison Gopnik** - Her talk was titled "The Golem vs. Stone Soup: Understanding How Children Learn Can Help Us Understand And Improve AI" (delivered on the opening day)

2. **Fei-Fei Li** - Her talk was titled "From Seeing to Doing: Ascending the Ladder of Visual Intelligence" (delivered on December 11, 2024)

3. **Lidong Zhou** - His talk was titled "A Match Made in Silicon: The Co-Evolution of Systems and AI" (delivered on December 12, 2024)

The initial search also mentioned **Sepp Horchreiter** and **Danica Kragic** as confirmed keynote speakers, though I didn't find their specific talk titles or schedules in the detailed results.

These invited talks covered diverse topics from cognitive science and child development (Gopnik), computer vision and AI (Li), to AI systems and infrastructure (Zhou), reflecting NeurIPS's interdisciplinary approach to machine learning and AI research.

## Inspect the state
As the conversation is now persisted in disk and each change of state is logged, we can retrospectively view the history of the states. This includes the user messages, the tool calls, and the answers from the LLM.

In [15]:
snapshot = graph.get_state(config)
pprint(snapshot.values["messages"])

[HumanMessage(content='Where was the latest NeurIPS conference held?', additional_kwargs={}, response_metadata={}, id='98ed59e8-912f-4631-9df1-462708890d53'),
 AIMessage(content=[{'text': "I'll search for information about the latest NeurIPS conference location.", 'type': 'text'}, {'id': 'toolu_01PRVV4QgQ3jY5eaN8jvcK6u', 'input': {'query': 'latest NeurIPS conference 2024 location where held', 'search_depth': 'basic'}, 'name': 'tavily_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_016h4xBDjzbqnDKG2tovuxMs', 'model': 'claude-sonnet-4-20250514', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 2128, 'output_tokens': 101, 'server_tool_use': None, 'service_tier': 'standard', 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}, 'model_name': 'claude-sonnet-4-20250514'}, id='run--f4825bb9-b4fa-4e6f-b2d3-afcd4a02cfb6-0', tool_calls=[{'na

If the graph is being executed, the `next` attribute tells us the node that will be executed next. As the graph has now finished execution, this attribute is an empty tuple.

In [16]:
snapshot.next

()